# Сетевой анализ дискурса постгуманистической философии

* ### О чём работа?
В этом исследовании Digital Humanities мне бы хотелось изучить поле мыслителей, в той или иной степени относимых к одному из самых актуальных философских направлений конца XX - начала XXI вв. — *постгуманизму*. Любое научное дискурсивное поле состоит из авторов (тех, кто в нём говорит), а также их способов говорить (в данном случае речь о книгах, статьях и прочих видах текстов), и история философии работает с множеством таких дискурсивных полей: к примеру, поле неоплатоников, поле немецких идеалистов, поле русских христианских философов и т.д. Однако проблема состоит в том, что мы не всегда способны подняться на "уровень птичьего полёта" и посмотреть, кто на кого ссылается в дискурсе, кто кого цитирует, в каких книгах и т.д. просто из-за того, что информации очень много. Поэтому для любого историка философии хорошим инструментом в работе является **визуализация**, с помощью которой он может видеть всю сеть целиком и делать выводы о влиятельности того или иного исследователя в его поле по соответствующим метрикам. Это исследование предлагает провести ту же самую задачу в поле постгуманистической философии — визуализировать постгуманистический дискурс при помощи сетевого анализа. Результаты могут очень пригодиться философам, историкам современной философии и всем, кто хочет ознакомиться с постгуманизмом.

* ### Цель работы
Методом сетевого анализа составить систему цитирований в постгуманистической философии и определить, кто является ключевыми авторами в постгуманизме.

* ### Задачи работы
1) Составить датасет авторов, занимавшихся постгуманистической философией или связанных с ней. В подготовке датасета я опирался на обзорные книги Дмитрия Хаустова *"Тёмные теории"* (2026) и Кэри Вулф *"What is Posthumanism?"* (2010), а также статью о постгуманизме в Википедии (https://en.wikipedia.org/wiki/Posthumanism). Собрано 23 автора и 44 книги.
2) При помощи регулярных выражений, библиотеки `pymupdf` и модуля `fitz` найти библиографические разделы в книгах и вычленить оттуда цитирования, которые потом станут рёбрами в графе.
3) Из собранной таблицы с данными сформировать граф и сделать общие выводы.

P.S. При имеющимся датасете этот код теоретически может быть применим к **любому дискурсивному полю** с учётом нескольких условий: 
- каждый PDF-файл из датасета имеет распознаваемый текст;
- в каждой книге есть библиографический раздел;
- каждый автор хотя бы раз либо цитировал кого-то из других авторов в датасете, либо был процитирован ими (иначе автор выпадет из сети). 

## Шаг 1. Импорт библиотек

Импортируем все библиотеки, необходимые для работы. Комментарии ниже.

In [20]:
import pymupdf # основная библиотека PyMuPDF для работы с PDF-файлами

import os # путь к файлам
import re # регулярные выражения

import pandas as pd # табличные данные
import networkx as nx # графы
from pyvis.network import Network # визуализация графов в HTML
import matplotlib.pyplot as plt # визуализация

## Шаг 2. Составление словаря авторов (ключ - фамилия по-английски, значение - имя на русском)

Переменная `folder` хранит название папки с датасетом, а `AUTHORS` — словарь с именами и фамилиями философов.

### Выбор датасета.
На этом этапе скажу о выборе авторов и их текстов для исследования. Опираясь на вышеуказанные источники, особенно на *"Тёмные теории"* Д. Хаустова, я постарался собрать наиболее важных авторов для постгуманизма, а у них — самые цитируемые работы. Разумеется, список не исчерпаем и может быть дополнен другими философами и менее цитируемыми текстами, однако у нашего исследования есть технический предел. Кроме того, в списке философов присутствуют те, которые не являются постгуманистами в строгом смысле, но которые оказали наибольшее влияние на это направление: постструктуралист Жиль Делёз, философы техники Бернар Стиглер и Жильбер Симондон и всем известный экзистенциалист Мартин Хайдеггер. Без этих мыслителей картина постгуманистического дискурса была бы неполной и разреженной.

In [11]:
folder = "project_posthumanism"

AUTHORS = {
    "latour": "Б. Латур",
    "stiegler": "Б. Стиглер",
    "colebrook": "К. Колбрук",
    "wolfe": "К. Вулф",
    "haraway": "Д. Харауэй",
    "thacker": "Ю. Такер",
    "ferrando": "Ф. Феррандо",
    "deleuze": "Ж. Делёз",
    "harman": "Г. Харман",
    "simondon": "Ж. Симондон",
    "grant": "И.Г. Грант",
    "bennett": "Дж. Беннетт",
    "barad": "К. Барад",
    "delanda": "М. ДеЛанда",
    "heidegger": "М. Хайдеггер",
    "hayles": "Н. Кэтрин Хейлс",
    "land": "Н. Ланд",
    "meillassoux": "К. Мейясу",
    "braidotti": "Р. Брайдотти",
    "brassier": "Р. Брассье",
    "negarestani": "Р. Негарестани",
    "ligotti": "Т. Лиготти",
    "morton": "Т. Мортон",
}

## Шаг 3. Парсинг имени файла

In [32]:
def parse_filename(a):
    # Отделяем .pdf в названии
    name = a.rsplit(".", 1)[0]

    # Отделяем год, если он есть в скобках 
    match = re.search(r"\((\d{4})\)\s*$", name)
    if match:
        # Ищем число из 4-х цифр, затем убираем его вместе со скобками из name
        year = match.group(1)
        name = name.replace(match.group(0), "").strip()
    else:
        year = None

    # Делим автора и название работы
    if " - " in name:
        authorname, title = name.split(" - ", 1)
    else:
        authorname, title = name, ""

    # Убираем пробелы, делим имя на слова, достаём последнее, приводим в нижний регистр — получаем фамилию
    surname = authorname.strip().split()[-1].lower()
    return surname, title.strip(), year

## Шаг 4. Функции для поиска цитирований

**Важная оговорка.** В этом проекте нет задачи рассматривать каждый текст вне библиографических разделов. На то есть две причины. Во-первых, существуют технические пределы и составить код, позволивший бы искать сноски, примечания и прочие дополнения любого текста, — достаточно непростая задача. Во-вторых, нам неважно, сколько раз один автор процитировал другого во всём тексте. Нужен сам факт влияния, который подтверждается хотя бы одним упоминанием имени в библиографии. Поэтому нас не интересует анализ точного количества цитирований одного автора другим, что требует, как я уже сказал, гораздо более сложного кода. Тем не менее я оцениваю уровень влияния по количеству книг (к примеру, если Харауэй упоминала Делёза в 2-х книгах, то вес ребра между ними будет больше — об этом подробнее в шаге 7).

In [44]:
# Список возможных названий библиографического раздела
BIB_HEADERS = ["bibliography", "references", "reference notes", "works cited", "notes", "literature"]

# 1) Склеиваем все страницы PDF в одну строку
def extract_text(path):
    doc = fitz.open(path) # как раз тут помогает PyMuPDF: он открывает PDF-файл и добавляет в doc весь текст
    pages_text = []
    for page in doc:
        pages_text.append(page.get_text())
    doc.close()
    # Раскидываем по слову на строчку
    full_text = "\n".join(pages_text)
    return full_text

# 2) Находим начало раздела библиографии и возвращаем текст от него до конца вместе с заголовком. Если ни того, ни другого нет — None.
# P.S. Мы не обойдёмся без доп. проверки названий библиографий. В некоторых текстах из датасета они пишутся как "B I B L I O G R A P H Y" 
# или "N O T E S", поэтому просто strip() здесь недостаточно. Будем очищать пробелы внутри каждого слова и проверять совпадение с BIB_HEADERS.
def get_bibliography(text):
    lines = text.split("\n")
    n = len(lines)
    for header in BIB_HEADERS:
        # Идём с конца книги, чтобы не попасться на ложные совпадения внутри текста
        for i in range(n - 1, -1, -1): 
            if re.sub(r"\s+", "", lines[i].lower()) == re.sub(r"\s+", "", header):
                # Считаем кол-во строк до начала библиографии (len(l) + 1 — длина слова + очищенный "\n"); выводим текст после start и название раздела
                start = sum(len(l) + 1 for l in lines[:i])
                return text[start:], header
    return None, None

# 3) Собираем всех авторов из списка AUTHORS в библиографическом разделе
TRICKY = {
    "grant": r"(?m)^\s*grant,",   # строка, начинающаяся с "Grant,"
    "land":  r"(?m)^\s*land,",
}

def find_quotes(text):
    low_text = text.lower()
    words = re.findall(r"\w+", low_text)
    word_set = set(words)
    found_names = set()
    for surname, russianname in AUTHORS.items():
        if surname in TRICKY:
            if re.search(TRICKY[surname], low_text):
                found_names.add(russianname)
        else:
            if surname in word_set:
                found_names.add(russianname)
    return found_names

## Шаг 5. Обработка файлов

In [45]:
# Создаём три списка: rows для таблицы с данными, failed_files для тех, что не получилось склеить; missing_texts для тех, где нет библиографии
rows, missing_texts, failed_files = [], [], []

# Смотрим, какой текст обрабатывается в данный момент; начинаем нумерацию с 1
for i, filename in enumerate(sorted(os.listdir(FOLDER)), 1):
    print(f"[{i}] В обработке: {filename}")
    path = os.path.join(FOLDER, filename)
    
    # Фамилия, название книги и год нужны для таблицы
    source, title, year = parse_filename(filename)
    # Но мы сразу меняем фамилию на русское имя для красоты
    source_name = AUTHORS[source]
    
    # Пробуем, что склеивание текста работает; если нет — добавляем в список failed_files
    try:
        text = extract_text(path)
    except Exception as e:
        print(f"Пропущен — ошибка чтения: {e}")
        failed_files.append((filename, str(e)))
        continue

    # Далее ищем библиографические разделы и их заголовки; если их тоже нет — добавляем текст в список missing_texts
    biblio_text, header = get_bibliography(text)
    if biblio_text is None:
        missing_texts.append(filename)
        continue

    # Собираем имена тех, кого автор процитировал
    cited = find_quotes(biblio_text)
    # Убираем самоцитирование
    if source_name in cited:
        cited.remove(source_name)

    # Собираем список словарей для таблицы
    for i in cited:
        rows.append({
            "Автор": source_name,
            "Название": title,
            "Год": year,
            "Кого цитировал": i,
            "Название раздела": header,
        })

df = pd.DataFrame(rows)
print(f"\nРёбер: {len(df)}")
print(f"Файлов без библиографии: {len(missing_bib)}")
print(f"Не удалось обработать: {len(failed_files)}\n")
df.head(15)

[1] В обработке: B. Latour - Laboratory Life (1987).pdf
[2] В обработке: B. Latour - Politics of Nature (2007).pdf
[3] В обработке: B. Latour - Reassembling the Social (2005).pdf
[4] В обработке: B. Latour - Science in Action (1987).pdf
[5] В обработке: B. Latour - We Have Never Been Modern (1991).pdf
[6] В обработке: B. Stiegler - Technics and Time, 1 (1994).pdf
[7] В обработке: B. Stiegler - Technics and Time, 2 (1996).pdf
[8] В обработке: C. Colebrook - Death of the PostHuman (2014).pdf
[9] В обработке: C. Wolfe - Animal Rites (2003).pdf
[10] В обработке: C. Wolfe - Before the Law (2012).pdf
[11] В обработке: C. Wolfe - What Is Posthumanism (2010).pdf
[12] В обработке: D. Haraway - A Cyborg Manifesto (1985).pdf
[13] В обработке: D. Haraway - Simians, Cyborgs and Women (1991).pdf
[14] В обработке: D. Haraway - Situated Knowledges (1988).pdf
[15] В обработке: D. Haraway - When Species Meet (2007).pdf
[16] В обработке: E. Thacker - In the Dust of This Planet (2011).pdf
[17] В обработке

,Автор,Название,Год,Кого цитировал,Название раздела
0,Б. Латур,Laboratory Life,1987,Д. Харауэй,references
1,Б. Латур,Laboratory Life,1987,М. Хайдеггер,references
2,Б. Латур,Reassembling the Social,2005,Д. Харауэй,bibliography
3,Б. Латур,Reassembling the Social,2005,Ж. Делёз,bibliography
4,Б. Латур,Science in Action,1987,К. Вулф,references
5,Б. Латур,We Have Never Been Modern,1991,Д. Харауэй,bibliography
6,Б. Латур,We Have Never Been Modern,1991,М. Хайдеггер,bibliography
7,Б. Латур,We Have Never Been Modern,1991,Ж. Делёз,bibliography
8,Б. Стиглер,"Technics and Time, 1",1994,М. Хайдеггер,bibliography
9,Б. Стиглер,"Technics and Time, 1",1994,Ж. Симондон,bibliography


## Шаг 6. Проверка наличия текстов без библиографических разделов

К сожалению, с этими текстами ничего не сделаешь, потому что в них отсутствует библиография. Но мы по-прежнему оставим их в датасете, потому что имена авторов всё равно нужны для поиска по библиографиям.

In [48]:
if missing_bib:
    print("В этих файлах не найден заголовок библиографии:")
    for f in missing_bib:
        print("-", f)
    print("\nДобавьте новые заголовки в BIB_HEADERS.")
else:
    print("Во всех файлах раздел библиографии найден!")

В этих файлах не найден заголовок библиографии:
- D. Haraway - A Cyborg Manifesto (1985).pdf
- G. Simondon - On the Mode of Existence of Technical Objects (1958).pdf
- M. Heidegger - The Question Concerning Technology, and Other Essays (1954).pdf
- N. Katherine Hayles - How We Became Posthuman (1999).pdf
- N. Katherine Hayles - Writing Machines (2002).pdf
- R. Braidotti - Nomadic Subjects (1994).pdf

Добавьте новые заголовки в BIB_HEADERS.


## Шаг 7. Построение графа

Мы строим направленный граф, в котором ребро-стрелка будет идти от одного автора к другому автору, которого первый процитировал. Кроме того, мы придадим каждому ребру вес в зависимости от количества книг, в которых есть пара цитирующего и цитируемого.

In [49]:
G = nx.DiGraph()

for _, row in df.iterrows():
    s, t = row["Автор"], row["Кого цитировал"]
    # Если между двумя авторами уже есть связь, то мы прибавляем к её весу + 1. 
    if G.has_edge(s, t):
        G[s][t]["weight"] += 1
    else:
        G.add_edge(s, t, weight=1)

print(f"Вершин в графе: {G.number_of_nodes()}")
print(f"Рёбер в графе: {G.number_of_edges()}")

Вершин в графе: 23
Рёбер в графе: 120


## Шаг 8. Вывод самых влиятельных авторов из списка

Наиболее влиятельных авторов для постгуманизма мы будем оценивать не по входящим рёбрам, а по известному алгоритму PageRank, изначально задуманному создателями Google. Если не вдаваться в формулу алгоритма, его суть в том, что узел важен, если на него ссылается много других важных узлов. Переводя на язык нашего исследования — цитирование от авторитетной, часто цитируемой, работы весит больше, чем от малозаметной, редко цитируемой.

In [50]:
# Формируем словарь с каждым автором и его входящими связями, т.е. сколько его цитировали 
# (причём сумма будет складываться не по кол-ву стрелок, а по их весу)
in_deg = dict(G.in_degree(weight="weight"))
pagerank = nx.pagerank(G)

most_noticeable = pd.DataFrame({
    "Кол-во цитирований": in_deg,
    "Влияние": pagerank,
}).fillna(0).sort_values("Влияние", ascending=False)

print("Самые влиятельные фигуры постгуманизма:")
most_noticeable.head(10)

Самые влиятельные фигуры постгуманизма:


,Кол-во цитирований,Влияние
М. Хайдеггер,28,0.180364
Ж. Делёз,30,0.136769
Ж. Симондон,8,0.094402
Д. Харауэй,16,0.063696
Б. Латур,16,0.058097
К. Вулф,7,0.041734
К. Мейясу,8,0.034935
Дж. Беннетт,8,0.034713
Н. Кэтрин Хейлс,7,0.034585
Р. Брассье,7,0.031191


## Шаг 9. Визуализация

Создаём граф на чёрном фоне с разноцветными узлами, белыми стрелками и именами философов на русском, написанными на чёрном фоне, чтобы они были читабельными.

In [51]:
# Простой набор из 12 цветов, по которому мы пройдём циклически и придадим каждому автору свой
palette = ["#ff0000", "#0000ff", "#00ff00", "#ffff00", "#524732", "#ffa500", "#800080", "#00ffff", "#ff00ff", "#92ada2", "#7fffd4", '#78f084']
authors_list = list(G.nodes())
node_colors = {name: palette[i % len(palette)] for i, name in enumerate(authors_list)}

# Атрибуты узлов графа: цвет, размер в зависимости от кол-ва входящих связей, title при наведении курсора
for node in G.nodes():
    deg = in_deg.get(node)
    G.nodes[node]["color"] = node_colors[node]
    G.nodes[node]["size"]  = 8 + deg * 1.2
    G.nodes[node]["title"] = (
        f"Имя: {node}\n"
        f"Входящих цитирований: {deg}\n"
        f"Влияние: {pagerank[node]:.3f}"
    )

# Раскрашиваем рёбра в белый цвет
for s, t in G.edges():
    G.edges[s, t]["color"] = "#ffffff" 

# Параметры самой сети
net = Network(
    height="800px",
    width="100%",
    bgcolor="#000000",
    font_color="#eeeeee",
    directed=True
)

net.from_nx(G)

# Настраиваем физику; к именам узлов добавляем чёрный фон для читабельности
net.set_options("""
{
  "physics": {
    "enabled": true,
    "barnesHut": {
      "gravitationalConstant": -8000,
      "centralGravity": 0.1,
      "springLength": 200,
      "springConstant": 0.01,
      "damping": 0.6
    }
  },
  "nodes": {
    "font": {
      "size": 18, "color": "#ffffff",
      "strokeWidth": 4, "strokeColor": "#000000"
    }
  }
}
""")

net.write_html("posthumanism_graph.html")

## Шаг 10. Выводы и перспективы

Сделаем несколько выводов о получившемся графе.
1) Мы видим двух ключевых фигур, которых цитируют чуть ли не все, а они никого не цитируют, поскольку жили раньше остальных исследователей - это Жиль Делёз и Мартин Хайдеггер, что довольно неудивительно, поскольку именно они являются самыми важными для всего постгуманизма. Заметно также, что третьим по величине философом по PageRank является Жильбер Симондон (0.09), не самый большой по in-degree на графе. 
2) Следом за Делёзом и Хайдеггером по величине узлов идут Бруно Латур и Донна Харауэй, одни из основателей постгуманистической теории. Как показывают веса рёбер-стрелок, они часто цитируют друг друга и их часто цитируют другие.
3) Наконец, по всему графу можно увидеть множество узлов меньше, это современные постгуманисты, последователи всех вышеназванных 4-х философов. Среди них есть 4 основателя спекулятивного реализма (К. Мейясу, Г. Харман, И. Грант и Р. Брассье), "тёмный эколог" Тимоти Мортон и др. Отдельно стоит указать такое связующее звено всего графа, Ф. Феррандо, исследовательницу постгуманизма, которая процитировала огромное число людей в списке, хотя при этом её не процитировал никто.

Какие у исследования были ограничения и какие перспективы можно наблюдать?
Во-первых, узкий датасет. Если бы нашей целью стояло исследование абсолютно всего направления постгуманистической философии, то стоило бы брать больше книг у каждого автора, а в лучшем случае — и больше книг, и больше авторов. Граф в этом исследовании указывает на общие характеристики интеллектуального поля постгуманизма, но они не так многогранны, как могли бы быть. Во-вторых, проблема оценки влияния. Как мы уже указывали ранее, нашей задачей было подтвердить факт влияния, а не его точный уровень. Для точной интерпретации авторитетности исследователей и влияния одного на другого, разумеется, имеет смысл написать код, который мог бы анализировать всё содержание каждой книги: точные цитаты в тексте, сноски, примечания, методологию и прочие дополнения, где могли бы быть указаны дополнительные авторы. В-третьих, как продолжение второго ограничения, отсутствие библиографий. А что делать, если библиографий нет вовсе? Тут данный проект приходит в тупик, потому что работает только с библиографическими разделами. Поэтому, к сожалению, погрешность оказалось неизбежной и, как выяснилось, в 6 текстах из датасета отсутствовала библиография или доп. примечания.

Итак, можно заключить, что это исследование вполне доступно для дальнейшего развития через: 1) увеличение датасета; 2) более глубокий анализ всего текста в каждой книге, а не только его частей; 3) исправление технических погрешностей (в случаях, если библиографии, примечания и пр. отсутствуют).